# 电路## 简介[电路类](../api/circuit.html) 表示具有任意拓扑结构的电路，该电路由任意数量的 N 端口 [网络](../api/network.html) 组成，这些网络相互连接。 类似于电子电路仿真器，该电路必须有一个（或多个）`Port` 连接到电路。 `Circuit` 对象允许检索 M 端口的 `Network`（以及因此的网络参数：$S$、$Z$ 等），其中 M 是定义的端口数量。 此外，`Circuit` 对象还允许计算整个电路的散射矩阵 $S$，即电路中各个连接点的“内部”散射矩阵。 计算算法基于参考文献 [[1](#ref1)]。下图说明了一个具有 2 个端口、`Network` 元素 $N_i$ 和连接点的网络：![通用电路](figures/circuit_general.svg)必须定义电路的连接列表（“网表”）。 该连接列表定义为一个由互连元组 `(network, port_number)` 组成的列表的列表：```connexions = [    [(network1, network1_port_nb), (network2, network2_port_nb), (network2, network2_port_nb), ...],    ...]```例如，用于构建上述电路的连接列表可以是：```connexions = [    [(port1, 0), (network1, 0), (network4, 0)],    [(network1, 1), (network2, 0), (network5, 0)],    [(network1, 2), (network3, 0)],    [(network2, 1), (network3, 1)],    [(network2, 2), (port2, 0)],    [(network5, 1), (ground1, 0)],    [(network5, 2), (open1, 0)]]```其中，我们假设 `port1`、`port2`、`ground1`、`open1` 以及所有 `network1` 到 `network5` 都是 scikit-rf [网络](../api/network.html) 对象，并且具有相同的 `Frequency`。 网络可以具有不同的（实数）特性阻抗：会考虑阻抗失配。 提供了方便的方法来创建端口和接地连接：* [Circuit.Port()](../api/generated/skrf.circuit.Circuit.Port.html#skrf.circuit.Circuit.Port)* [Circuit.Ground()](../api/generated/skrf.circuit.Circuit.Ground.html#skrf.circuit.Circuit.Ground)* [Circuit.SeriesImpedance()](../api/generated/skrf.circuit.Circuit.SeriesImpedance.html#skrf.circuit.Circuit.SeriesImpedance)* [Circuit.ShuntAdmittance()](../api/generated/skrf.circuit.Circuit.ShuntAdmittance.html#skrf.circuit.Circuit.ShuntAdmittance)* [Circuit.Open()](../api/generated/skrf.circuit.Circuit.Open.html#skrf.circuit.Circuit.Open)

**警告**[Circuit](https://scikit-rf.readthedocs.io/en/latest/api/circuit.html) 要求每个 `Network` 都有不同的 `Network` 名称。因此，如果您想多次使用同一个 `Network`，则必须复制该 `Network`（例如，使用 `.copy()` 方法），并为每个副本指定不同的 `.name` 属性，如下所示：```# 假设 network1 是您想要复制的 Network：network2 = network1.copy()network1.name = 'My first Network'network2.name = 'My second network'```此外，`(network, port_number)` 组合在连接列表中应只出现一次。如果您想将多个网络连接到同一个网络（例如，将多个网络接地），则有两种解决方案：* 将 N 个网络一次性连接到单个 `Ground` 对象，如下所示：```# 在连接列表中：...    [(network1,1),(network2,1),(network3,1),(gnd,0)]...```* 或者，根据需要创建多个不同的 `Ground` 对象（同样，每个对象都具有唯一的 `name` 属性，如上所述）。```# 在连接列表中：...    [(network1,1), (gnd1,0)],    [(network2,1), (gnd2,0)],    [(network3,1), (gnd3,0)],...```

一旦定义了连接列表，就可以使用以下代码创建 `Circuit`：```resulting_circuit = rf.circuit.Circuit(connexions)````resulting_circuit` 是一个 [Circuit](../api/circuit.html) 对象。通过 [Circuit.network](../api/generated/skrf.circuit.Circuit.network.html#skrf.circuit.Circuit.network) 参数，可以获得最终的 2 端口 `Network`：```resulting_network = resulting_circuit.network```

请注意，也可以使用 `scikit-rf` 的 [连接方法](../api/network.html#connecting-networks) 手动创建由多个 `Network` 对象组成的电路。 尽管使用 `Circuit` 方法构建多个 `Network` 可能会显得比“经典”方法构建电路更繁琐，但随着电路复杂性的增加，特别是当元件并联连接时，`Circuit` 方法就显得很有用，因为它提高了代码的可读性。 此外，可以使用其 `plot_graph` 方法绘制 `Circuit` 电路拓扑图，这对于快速检查电路是否按照预期构建非常有用。

## 示例### 带有负载的传输线假设有一根 50Ω 无损传输线，其末端连接了一个阻抗为 $Z_L = 75\Omega$ 的负载。![带有负载的传输线电路](figures/circuit_loaded_transmission_line1.svg)如果传输线的电长度为 $\theta = 0$，那么我们可以预期反射系数为：$$\rho = s = \frac{Z_L - Z_0}{Z_L + Z_0} = 0.2$$

In [ ]:
import skrf as rf
from skrf.circuit import Circuit

rf.stylely()

In [ ]:
Z_0 = 50
Z_L = 75
theta = 0

# the necessary Frequency description
freq = rf.Frequency(start=1, stop=2, unit='GHz', npoints=3)

# The combination of a transmission line + a load can be created
# using the convenience delay_load method
# important: all the Network must have the parameter "name" defined
tline_media = rf.media.DefinedGammaZ0(freq, z0=Z_0)
delay_load = tline_media.delay_load(rf.tlineFunctions.zl_2_Gamma0(Z_0, Z_L), theta, unit='deg', name='delay_load')

# the input port of the circuit is defined with the Circuit.Port method
port1 = Circuit.Port(freq, 'port1', z0=Z_0)

# connection list
cnx = [
    [(port1, 0), (delay_load, 0)]
]
# building the circuit
cir = Circuit(cnx)

# getting the resulting Network from the 'network' parameter:
ntw = cir.network
print(ntw)

In [ ]:
# as expected the reflection coefficient is:
print(ntw.s[0])

也可以使用串联阻抗网络构建上述电路，然后进行短路：![负载传输线电路：第二个版本](figures/circuit_loaded_transmission_line2.svg)为此，需要使用 `Ground()` 方法来生成所需的 `Network` 对象。

In [ ]:
port1 = Circuit.Port(freq, 'port1', z0=Z_0)
# piece of transmission line and series impedance
trans_line = tline_media.line(theta, unit='deg', name='trans_line')
load = tline_media.resistor(Z_L, name='delay_load')
# ground network (short)
ground = Circuit.Ground(freq, name='ground')

# connection list
cnx = [
    [(port1, 0), (trans_line, 0)],
    [(trans_line, 1), (load, 0)],
    [(load, 1), (ground, 0)]
]
# building the circuit
cir = Circuit(cnx)
# the result if the same :
print(cir.network.s[0])


### LC 滤波器这里我们模拟一个低通 LC 滤波器，并使用以下来自 [rf-tools.com](https://rf-tools.com/lc-filter/) 的示例值：![低通滤波器](figures/circuit_filter1.svg)

In [ ]:
freq = rf.Frequency(start=0.1, stop=10, unit='GHz', npoints=1001)
tl_media = rf.media.DefinedGammaZ0(freq, z0=50, gamma=1j*freq.w/rf.constants.c)
C1 = tl_media.capacitor(3.222e-12, name='C1')
C2 = tl_media.capacitor(82.25e-15, name='C2')
C3 = tl_media.capacitor(3.222e-12, name='C3')
L2 = tl_media.inductor(8.893e-9, name='L2')
RL = tl_media.resistor(50, name='RL')
gnd = Circuit.Ground(freq, name='gnd')
port1 = Circuit.Port(freq, name='port1', z0=50)
port2 = Circuit.Port(freq, name='port2', z0=50)

cnx = [
    [(port1, 0), (C1, 0), (L2, 0), (C2, 0)],
    [(L2, 1), (C2, 1), (C3, 0), (port2, 0)],
    [(gnd, 0), (C1, 1), (C3, 1)],
]
cir = Circuit(cnx)
ntw = cir.network

In [ ]:
ntw.plot_s_db(m=0, n=0, lw=2, logx=True)
ntw.plot_s_db(m=1, n=0, lw=2, logx=True)

在构建由少量网络组成的“电路”时，以图形方式表示连接可能有助于检查潜在的错误。可以使用 `Circuit.plot_graph()` 方法来实现这一点。端口用三角形表示，网络用正方形表示，互连用圆形表示。可以显示网络名称以及与其关联的端口（和特性阻抗）：

In [ ]:
cir.plot_graph(network_labels=True, network_fontsize=15,
               port_labels=True, port_fontsize=15,
              edge_labels=True, edge_fontsize=10)

### 电路简化这里，我们对一个稍微复杂一些的带通 LC 滤波器进行建模，以演示电路简化，示例值取自 [markimicrowave.com](https://markimicrowave.com/technical-resources/tools/lc-filter-design-tool/)：![带通滤波器](figures/circuit_filter2.svg)

In [ ]:
freq = rf.Frequency(50, 200, 301, 'mhz')
tl_media = rf.media.DefinedGammaZ0(frequency=freq, z0=50)
gnd = Circuit.Ground(frequency=freq,name='ground')
C1 = tl_media.capacitor(6.353e-12, name='C1')
L1 = tl_media.inductor(402.7e-9, name='L1')
C2 = tl_media.capacitor(61.08e-12, name='C2')
L2 = tl_media.inductor(13.63e-9, name='L2')
C3 = tl_media.capacitor(187.8e-12, name='C3')
L3 = tl_media.inductor(41.89e-9, name='L3')
C4 = tl_media.capacitor(6.353e-12, name='C4')
L4 = tl_media.inductor(402.27e-9, name='L4')
Port1 = Circuit.Port(frequency=freq, name='PortIn')
Port2 = Circuit.Port(frequency=freq, name='PortOut')
cnx = [
    [(Port1, 0), (C1, 0)],
    [(C1, 1), (L1, 0)],
    [(L1, 1), (C2, 0), (C3, 0), (C4, 0)],
    [(C2, 1), (L2, 0)],
    [(C3, 1), (L3, 0)],
    [(L2, 1), (L3, 1), (gnd, 0)],
    [(C4, 1), (L4, 0)],
    [(L4, 1), (Port2, 0)]
]
cir = Circuit(cnx)
ntw = cir.network

In [ ]:
ntw.plot_s_db(m=0, n=0, lw=2)
ntw.plot_s_db(m=1, n=0, lw=2)

与前面的示例一样，我们可以使用 `Circuit.plot_graph()` 方法以图形方式表示电路连接。

In [ ]:
cir.plot_graph(network_labels=True, network_fontsize=15,
               port_labels=True, port_fontsize=15,
              edge_labels=True, edge_fontsize=10)

当我们仅关注“端口”并忽略内部“网络”时，我们可以使用 `rf.reduce_circuit()` 来简化电路。此方法将自动使用 `rf.connect()` 方法将相邻的网络串联连接起来，并仅保留“端口”以及包含 2 个以上“网络”的连接，以便进行电路分析。

In [ ]:
reduced_cnx = rf.circuit.reduce_circuit(cnx)
reduced_cir = Circuit(reduced_cnx)
reduced_cir.plot_graph(network_labels=True, network_fontsize=15,
               port_labels=True, port_fontsize=15,
              edge_labels=True, edge_fontsize=10)

此外，可选参数可以影响电路简化的行为。例如，`split_ground=True` 表示将电路中共享的“地”分成两部分：

In [ ]:
fully_reduced_cnx = rf.circuit.reduce_circuit(cnx, split_ground=True)
fully_reduced_cir = Circuit(fully_reduced_cnx)

fully_reduced_cir.plot_graph(network_labels=True, network_fontsize=15,
               port_labels=True, port_fontsize=15,
              edge_labels=True, edge_fontsize=10)

最后，`Circuit` 构造函数提供了一个自动简化电路的选项。将 `auto_reduce` 设置为 `True` 可以创建一个高度简化的 `Circuit` 对象，从而提高端口的散射参数计算效率。

In [ ]:
from timeit import timeit

import numpy as np

fully_reduced_ntw = fully_reduced_cir.network
auto_reduced_cir = Circuit(cnx, auto_reduce=True)
auto_reduced_ntw = auto_reduced_cir.network
n_times = 10
print("Unreduced Circuit calculate Network in "
      f"{timeit(lambda: cir.network, number=n_times)/n_times:.4f}s")
print("Auto-reduced Circuit calculate Network in "
      f"{timeit(lambda: auto_reduced_cir.network, number=n_times)/n_times:.4f}s")

print("Manually reduced circuit has the same s-parameters as the original circuit: "
      f"{np.allclose(ntw.s, fully_reduced_ntw.s)}")
print("Auto reduced circuit has the same s-parameters as the original circuit: "
      f"{np.allclose(ntw.s, auto_reduced_ntw.s)}")

## References<div id="ref1"></div>[1] Hallbjörner, P., 2003. Method for calculating the scattering matrix of arbitrary microwave networks giving both internal and external scattering. Microw. Opt. Technol. Lett. 38, 99–102. https://doi.org/10/d27t7m